# Side-by-Side Model Comparison

Compare latency and response quality between two models hosted in Azure AI Foundry.

**Latency metrics:** TTFT, Total Time, Tokens/sec  
**Quality evaluation:** LLM-as-judge (optional)  
**API support:** Azure AI Inference API or OpenAI Responses API

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Inference API (azure-ai-inference)
from utils import get_client, run_latency_test, judge_comparison
# Responses API (openai SDK)
from utils import get_openai_client, run_responses_latency_test, judge_responses_comparison

load_dotenv()

## 2. Configure API Type & Models

Choose which API to use:
- **`"responses"`** — OpenAI Responses API (recommended for OpenAI models, also works with Grok, Mistral, Llama)
- **`"inference"`** — Azure AI Inference API

Reads model names from `.env` by default. Override inline to test different combinations.

In [ ]:
# Choose API type: "responses" or "inference"
API_TYPE = "responses"

if API_TYPE == "responses":
    client = get_openai_client()
    _run_test = run_responses_latency_test
    _judge = judge_responses_comparison
else:
    client = get_client()
    _run_test = run_latency_test
    _judge = judge_comparison

MODEL_A = os.getenv("AZURE_MODEL_A", "gpt-5.2")
MODEL_B = os.getenv("AZURE_MODEL_B", "gpt-5.4-mini")
JUDGE_MODEL = os.getenv("AZURE_JUDGE_MODEL", "gpt-5.2")

print(f"API:     {API_TYPE}")
print(f"Model A: {MODEL_A}")
print(f"Model B: {MODEL_B}")
print(f"Judge:   {JUDGE_MODEL}")

## 3. Define Test Prompts

Both models receive the same prompts. Mix short and long expected outputs for varied measurement.

In [ ]:
prompts = [
    "What is the capital of France?",
    "Explain quantum computing in 3 sentences.",
    "Write a Python function that checks if a string is a palindrome.",
    "Summarize the key ideas of machine learning in a short paragraph.",
]

## 4. Run Latency Tests

In [ ]:
print(f"Testing {MODEL_A} via {API_TYPE} API...")
results_a = _run_test(client, MODEL_A, prompts)
print(f"  ✓ {len(results_a)} prompts completed")

print(f"Testing {MODEL_B} via {API_TYPE} API...")
results_b = _run_test(client, MODEL_B, prompts)
print(f"  ✓ {len(results_b)} prompts completed")

## 5. Side-by-Side Latency Comparison

In [ ]:
df_a = pd.DataFrame(results_a)
df_b = pd.DataFrame(results_b)
df_all = pd.concat([df_a, df_b], ignore_index=True)

comparison = df_all.pivot_table(
    index="prompt_index",
    columns="model",
    values=["ttft", "total_time", "tokens_per_second", "completion_tokens"],
)
comparison

In [ ]:
summary = df_all.groupby("model").agg(
    avg_ttft=("ttft", "mean"),
    avg_total_time=("total_time", "mean"),
    avg_tokens_per_sec=("tokens_per_second", "mean"),
    total_tokens=("completion_tokens", "sum"),
).round(4)

print("=== Latency Summary ===")
summary

## 6. Latency Charts

In [ ]:
metrics = ["ttft", "total_time", "tokens_per_second"]
titles = ["Time to First Token (s)", "Total Response Time (s)", "Tokens per Second"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric, title in zip(axes, metrics, titles):
    pivot = df_all.pivot(index="prompt_index", columns="model", values=metric)
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Prompt")
    ax.legend(title="Model")

plt.tight_layout()
plt.show()

## 7. LLM-as-Judge Quality Evaluation (Optional)

Uses a separate judge model to evaluate response quality on configurable criteria.  
**Skip this section** if you only need latency comparison.

In [ ]:
# Set to True to run the judge evaluation
RUN_JUDGE = True

# Customize criteria as needed
CRITERIA = ["Accuracy", "Completeness", "Clarity"]

In [ ]:
if RUN_JUDGE:
    print(f"Running judge ({JUDGE_MODEL}) via {API_TYPE} API on {len(prompts)} prompt pairs...")
    judgments = _judge(
        client=client,
        judge_model=JUDGE_MODEL,
        prompts=prompts,
        results_a=results_a,
        results_b=results_b,
        model_a_name=MODEL_A,
        model_b_name=MODEL_B,
        criteria=CRITERIA,
    )
    print(f"  ✓ {len(judgments)} evaluations completed")
else:
    print("Judge evaluation skipped. Set RUN_JUDGE = True to enable.")

In [ ]:
if RUN_JUDGE:
    # Build a summary table of judge results
    judge_rows = []
    for j in judgments:
        row = {"prompt": j["prompt"], "winner": j["overall_winner"], "reasoning": j["overall_reasoning"]}
        for ev in j["evaluations"]:
            row[f"{ev['criterion']}_A"] = ev["score_a"]
            row[f"{ev['criterion']}_B"] = ev["score_b"]
        judge_rows.append(row)

    df_judge = pd.DataFrame(judge_rows)
    print(f"=== Quality Evaluation ({MODEL_A} vs {MODEL_B}) ===")
    df_judge

In [ ]:
if RUN_JUDGE:
    wins = df_judge["winner"].value_counts()
    print("=== Win/Loss/Tie Summary ===")
    for outcome, count in wins.items():
        label = MODEL_A if outcome == "A" else MODEL_B if outcome == "B" else "Tie"
        print(f"  {label}: {count}")

## 8. Combined View

Responses side by side for manual inspection.

In [ ]:
for i, prompt in enumerate(prompts):
    print(f"\n{'='*80}")
    print(f"Prompt {i}: {prompt}")
    print(f"{'='*80}")
    print(f"\n--- {MODEL_A} (TTFT: {results_a[i]['ttft']}s, Total: {results_a[i]['total_time']}s) ---")
    print(results_a[i]["response_text"][:500])
    print(f"\n--- {MODEL_B} (TTFT: {results_b[i]['ttft']}s, Total: {results_b[i]['total_time']}s) ---")
    print(results_b[i]["response_text"][:500])